In [2]:
import pandas as pd
import numpy as np
import surprise
from surprise.model_selection import train_test_split

In [3]:
df = pd.read_csv("../Cases_Rec_Sys/Amazon Movie Ratings/Amazon.csv")

ratings = pd.melt(df, id_vars='user_id', 
                  var_name="item_id",value_name="rating")

ratings = ratings[ratings['rating'].notna()]

In [4]:
ratings

,user_id,item_id,rating
0,A3R5OBKS7OM2IR,Movie1,5.0
4848,A3R5OBKS7OM2IR,Movie2,5.0
9697,AH3QC2PC1VTGP,Movie3,2.0
14546,A3LKP6WPMP9UKX,Movie4,5.0
14547,AVIY68KEPQ5ZD,Movie4,5.0
...,...,...,...
998683,A1IMQ9WMFYKWH5,Movie206,5.0
998684,A1KLIKPUF5E88I,Movie206,5.0
998685,A5HG6WFZLO10D,Movie206,5.0
998686,A3UU690TWXCG1X,Movie206,5.0


In [5]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings["rating"].max()
lowest_rating, highest_rating

(np.float64(1.0), np.float64(5.0))

In [6]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

similarity_options  = {"name":"cosine","user_based":True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())



Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
from surprise.model_selection import train_test_split




In [8]:
param_grid = {"n_epochs":[50],
              'n_factors':[100,50],
             'reg_all':np.linspace(0.01,0.8,5)}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD,param_grid,measures = ["rmse","mae"],cv=kfold, joblib_verbose=3,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    2.0s remaining:    0.7s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    2.1s finished


In [9]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

1.1575149055362046
{'n_epochs': 50, 'n_factors': 50, 'reg_all': np.float64(0.01)}


In [19]:
param_grid = {"n_epochs":[50],
              'n_factors':[100,50],
             'reg_all':np.linspace(0.01,0.8,5)}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.KNNWithZScore,param_grid,measures = ["rmse","mae"],cv=kfold, joblib_verbose=3,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    4.2s remaining:    1.6s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    4.9s finished


In [20]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

1.2018142692025235
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.01)}


In [21]:
param_grid = {"n_epochs":[50],
              'n_factors':[100,50],
             'reg_all':np.linspace(0.01,0.8,5)}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.KNNBasic,param_grid,measures = ["rmse","mae"],cv=kfold, joblib_verbose=3,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    2.1s remaining:    0.7s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    2.6s finished


In [22]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

1.1982508756078385
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.01)}


In [10]:
# Building bets model on whole data
algo =gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

In [12]:
iids = ratings["item_id"].unique()
iids

array(['Movie1', 'Movie2', 'Movie3', 'Movie4', 'Movie5', 'Movie6',
       'Movie7', 'Movie8', 'Movie9', 'Movie10', 'Movie11', 'Movie12',
       'Movie13', 'Movie14', 'Movie15', 'Movie16', 'Movie17', 'Movie18',
       'Movie19', 'Movie20', 'Movie21', 'Movie22', 'Movie23', 'Movie24',
       'Movie25', 'Movie26', 'Movie27', 'Movie28', 'Movie29', 'Movie30',
       'Movie31', 'Movie32', 'Movie33', 'Movie34', 'Movie35', 'Movie36',
       'Movie37', 'Movie38', 'Movie39', 'Movie40', 'Movie41', 'Movie42',
       'Movie43', 'Movie44', 'Movie45', 'Movie46', 'Movie47', 'Movie48',
       'Movie49', 'Movie50', 'Movie51', 'Movie52', 'Movie53', 'Movie54',
       'Movie55', 'Movie56', 'Movie57', 'Movie58', 'Movie59', 'Movie60',
       'Movie61', 'Movie62', 'Movie63', 'Movie64', 'Movie65', 'Movie66',
       'Movie67', 'Movie68', 'Movie69', 'Movie70', 'Movie71', 'Movie72',
       'Movie73', 'Movie74', 'Movie75', 'Movie76', 'Movie77', 'Movie78',
       'Movie79', 'Movie80', 'Movie81', 'Movie82', 'Movie83'

In [15]:
ratings["user_id"].unique()

array(['A3R5OBKS7OM2IR', 'AH3QC2PC1VTGP', 'A3LKP6WPMP9UKX', ...,
       'A5HG6WFZLO10D', 'A3UU690TWXCG1X', 'AI4J762YI6S06'],
      shape=(4848,), dtype=object)

In [17]:
user = "A3R5OBKS7OM2IR"
u_iid = ratings[ratings['user_id']==user]['item_id'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 ['Movie1' 'Movie2']

No of items rated by user A3R5OBKS7OM2IR : 2 

Items not rated by the user or those items for which the expected ratings are to be predicted :  ['Movie10' 'Movie100' 'Movie101' 'Movie102' 'Movie103' 'Movie104'
 'Movie105' 'Movie106' 'Movie107' 'Movie108' 'Movie109' 'Movie11'
 'Movie110' 'Movie111' 'Movie112' 'Movie113' 'Movie114' 'Movie115'
 'Movie116' 'Movie117' 'Movie118' 'Movie119' 'Movie12' 'Movie120'
 'Movie121' 'Movie122' 'Movie123' 'Movie124' 'Movie125' 'Movie126'
 'Movie127' 'Movie128' 'Movie129' 'Movie13' 'Movie130' 'Movie131'
 'Movie132' 'Movie133' 'Movie134' 'Movie135' 'Movie136' 'Movie137'
 'Movie138' 'Movie139' 'Movie14' 'Movie140' 'Movie141' 'Movie142'
 'Movie143' 'Movie144' 'Movie145' 'Movie146' 'Movie147' 'Movie148'
 'Movie149' 'Movie15' 'Movie150' 'Movie151' 'Movie152' 'Movie153'
 'Movie154' 'Movie155' 'Movie156' 'Movie157' 'Movie158' 'Movie159'
 'Movie16' 'Movie160' 'Movie161' 'Movie162' 'Movie163' 'Movie164'
 'Movi

In [18]:
testset = [[user,iid,0.] for iid in iids_ro_pred]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
2,Movie101,5.000000
9,Movie108,5.000000
20,Movie118,5.000000
22,Movie12,5.000000
69,Movie162,5.000000
63,Movie157,5.000000
142,Movie43,5.000000
195,Movie91,5.000000
117,Movie206,5.000000
126,Movie29,5.000000
